In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import integrate
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import csv, json

In [3]:
ls ../RawData_Siglent/

siglent.1766046701688.CH1.csv  siglent.1766046701688.CH1.zip*


In [7]:
EXPERIMENT = "stress-ng"
#EXPERIMENT = "sleep"

if EXPERIMENT=="stress-ng":
    test = "stress-ng"
    input_directory = '../RawData_Siglent/'
    output_directory = '../ChunkedData_Siglent/'
    # From the script homemade_script_energylogger_overhead_stressng.sh we can deduct the order
    #for load in 0 20 40 60 80 100
    # do
    #	for j in 1 2 3 4 5 6 7 8 9 10    
    #execution_order =  [0]*100+[20]*100+[40]*100+[60]*100+[80]*100+[100]*100
    
elif EXPERIMENT == "sleep":
    test = "sleep"
    input_directory = '../RawData_Siglent/4-dec/'
    output_directory = '../ChunkedData_Siglent/'
    execution_order =  [0]*11*35
else:
    print("error")

In [9]:
os.listdir(input_directory)
onlydir = [f for f in os.listdir(input_directory) if os.path.isdir(os.path.join(input_directory, f))]
onlyfiles = [f for f in os.listdir(input_directory) if os.path.isfile(os.path.join(input_directory, f))]

In [11]:
onlyfiles

['.DS_Store', 'siglent.1766046701688.CH1.csv', 'siglent.1766046701688.CH1.zip']

In [13]:
def parse_csv_line(line: str):
    # csv.reader handles the quoted JSON payload with doubled quotes
    return next(csv.reader([line], delimiter=',', quotechar='"', doublequote=True))

In [15]:
def is_json(perhaps_json):
  try:
    json.loads(perhaps_json)
  except ValueError as e:
    return False
  return True


In [21]:
dfs_list = []

dirs = [f for f in os.listdir(input_directory) if os.path.isdir(os.path.join(input_directory, f))]
directory = "."
#for directory in dirs:
execution_number = 0
for input_file in os.listdir(input_directory+'/'+directory):
    file_path = input_directory+'/'+directory+'/'
    file_name = os.path.basename(input_file)
    save_lines = False
    if file_name.endswith('.csv'):
        with open(file_path+file_name) as f:
            for line in f:
            # Do something with 'line'        
            #line = msg_line
                # 1765540947109,CH1,MESSAGE,"{""channelId"":""CH1"",""message"":""start,sigmark,1,0,0"",""timestamp"":1765540947109}"
                row =  parse_csv_line(line)
                if row[2] == 'MESSAGE': 
                    row_dict = json.loads(row[3])
                    if is_json(row_dict['message']):
                        message = json.load(row_dict['message'])
                    else:
                        message_list = row_dict['message'].split(',')
                        message = {}
                        # sh post_to_sigless.sh 192.168.50.101:8000 CH1 "stop,<computerID>,<class>"
                        message['action'] = message_list[0]
                        # message['call'] = message_list[1]
                        message['round'] = message_list[2]
                        message['load'] = message_list[3]
                        message['period'] = message_list[4]
                    #print(message)
                    #print(row_dict)
                    

                    if message['action'] =='start':
                        start_time = row_dict['timestamp']
                        message_values = list(message.values())+[str(start_time)]
                        file_addition = '_'.join(message_values[1:])
                        #print(file_addition)
                        
                        # Add information from message
                        message.pop('action')
                        start_row = message
                        start_row['start_time'] = start_time
                        start_row['load'] = message['load']
                        start_row['period'] = message['period']
                        start_row['round'] = message['round']
                        start_row['id'] = execution_number
                        execution_number += 1
                        start_row['channelId'] =  row_dict['channelId']
                        
                        outputfile = output_directory+file_addition+'_'+str(execution_number)+'_'+file_name
    
                        
                        if(not(save_lines)):
                            f_out = open(outputfile, "w")   # 'r' for reading and 'w' for writing                       
                            save_lines = True
                        else:
                            print(outputfile)
                            f_out = open(outputfile, "w")
        
                    elif message['action'] == 'stop':
                        end_time = row_dict['timestamp']
                        duration = end_time - start_time
                        start_row['exp_duration'] = duration
                        start_row['end_time'] = end_time
                        start_row['experiment'] = test
                        #start_row['load'] = execution_order[execution_number]
                        # print('meta_row=',start_row)
                        save_lines = False # No more lines for this flow  
                        f_out.close()   # Close output file
                        # save the meta data as the first two lines of the chunked CSV file
                        a = list(start_row.keys())
                        line1 = ','.join(a) 
                        a = list(start_row.values())
                        line2 = ','.join([str(s) for s in a])
                        with open(outputfile, 'r') as original:
                            data = original.read()
                        with open(outputfile, 'w') as modified:
                            modified.write(line1+"\n"+line2+"\n" + data)
                        
                            
                elif row[2] == 'POWER' and save_lines:  
                    f_out.write(line)   # Write line into the file 
            



{'action': 'start', 'round': '1', 'load': '0', 'period': '0'}
{'action': 'stop', 'round': '1', 'load': '0', 'period': '0'}
meta_row= {'round': '1', 'load': '0', 'period': '0', 'start_time': 1765540947109, 'id': 0, 'channelId': 'CH1', 'exp_duration': 80050, 'end_time': 1765541027159, 'experiment': 'stress-ng'}
{'action': 'start', 'round': '1', 'load': '0', 'period': '2'}
{'action': 'stop', 'round': '1', 'load': '0', 'period': '2'}
meta_row= {'round': '1', 'load': '0', 'period': '2', 'start_time': 1765541047677, 'id': 1, 'channelId': 'CH1', 'exp_duration': 80025, 'end_time': 1765541127702, 'experiment': 'stress-ng'}
{'action': 'start', 'round': '1', 'load': '0', 'period': '1'}
{'action': 'stop', 'round': '1', 'load': '0', 'period': '1'}
meta_row= {'round': '1', 'load': '0', 'period': '1', 'start_time': 1765541150221, 'id': 2, 'channelId': 'CH1', 'exp_duration': 80026, 'end_time': 1765541230247, 'experiment': 'stress-ng'}
{'action': 'start', 'round': '1', 'load': '0', 'period': '0.5'}
{'a